In [1]:
from pathlib import Path
import json, subprocess, sys

PIPELINE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PIPELINE_ROOT / "src"))
REPOSITORY_ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], cwd=PIPELINE_ROOT, text=True).strip())
BRANCH = subprocess.check_output(["git", "branch", "--show-current"], cwd=PIPELINE_ROOT, text=True).strip()
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PIPELINE_ROOT, text=True).strip()
SUMMARY_PATH = PIPELINE_ROOT / "runs/fixture_pipeline_summary.json"
SUMMARY = json.loads(SUMMARY_PATH.read_text(encoding="utf-8")) if SUMMARY_PATH.exists() else {}
METADATA = {
    "repositoryRoot": str(REPOSITORY_ROOT),
    "branch": BRANCH,
    "HEAD": HEAD,
    "contractVersion": "MISSING",
    "crawlReleaseId": "NONE",
    "dataVersion": SUMMARY.get("dataVersion", "NONE"),
    "asOfDate": "2026-08-06",
}
print(json.dumps(METADATA, ensure_ascii=False, indent=2))

{
  "repositoryRoot": "/home/sieg/projects-wsl/SBS_dataScience",
  "branch": "agent/p4-pipeline-v2",
  "HEAD": "cbd8b848a28201bac506a96797fc04cdb20b35e9",
  "contractVersion": "MISSING",
  "crawlReleaseId": "NONE",
  "dataVersion": "synthetic-fixture-v1",
  "asOfDate": "2026-08-06"
}


# Auxiliary similarity

This notebook never treats synthetic fixtures as source observations.

In [2]:
STAGE = '90AuxSimilarity'
NOTE = 'BLOCKED: auxiliary analysis awaits stable source-backed RQ1/RQ2 mart'
print(json.dumps({
    "stage": STAGE,
    "note": NOTE,
    "dataProvenance": SUMMARY.get("dataProvenance", "none"),
    "empiricalAnalysisAllowed": SUMMARY.get("empiricalAnalysisAllowed", False),
    "contractReady": SUMMARY.get("contract", {}).get("ready", False),
    "crawlReleaseCount": SUMMARY.get("crawlReleaseCount", 0),
    "fixtureRows": SUMMARY.get("rows", {}),
}, ensure_ascii=False, indent=2))

{
  "stage": "90AuxSimilarity",
  "note": "BLOCKED: auxiliary analysis awaits stable source-backed RQ1/RQ2 mart",
  "dataProvenance": "generated_structural_fixture",
  "empiricalAnalysisAllowed": false,
  "contractReady": false,
  "crawlReleaseCount": 0,
  "fixtureRows": {
    "raw": 6,
    "normalized": 6,
    "tracks": 6,
    "sections": 10,
    "requirements": 10,
    "labels": 6,
    "ncsUnits": 3,
    "ncsMatches": 4,
    "postingAnalysisMart": 6,
    "timeSeriesMart": 10,
    "excludedPostings": 1
  }
}


In [3]:
rows = SUMMARY.get("rows", {})
FINAL = {
    "inputRows": rows.get("raw", 0),
    "outputRows": rows.get("postingAnalysisMart", 0),
    "excludedRows": rows.get("excludedPostings", 0),
    "qualityStatus": "PASS_FIXTURE_ONLY" if SUMMARY.get("postingMartQuality", {}).get("passed") else "BLOCKED_NO_INPUT",
    "outputPaths": [item.get("path") for item in SUMMARY.get("artifactManifest", [])],
    "outputSha256": [item.get("sha256") for item in SUMMARY.get("artifactManifest", [])],
}
print(json.dumps(FINAL, ensure_ascii=False, indent=2))

{
  "inputRows": 6,
  "outputRows": 6,
  "excludedRows": 1,
  "qualityStatus": "PASS_FIXTURE_ONLY",
  "outputPaths": [
    "/home/sieg/projects-wsl/SBS_dataScience/DSJA/project_4/pipeline/data/marts/postingAnalysisMart.parquet",
    "/home/sieg/projects-wsl/SBS_dataScience/DSJA/project_4/pipeline/data/marts/timeSeriesMart.parquet",
    "/home/sieg/projects-wsl/SBS_dataScience/DSJA/project_4/pipeline/data/warehouse/p4.duckdb"
  ],
  "outputSha256": [
    "5447233502a4e3f8cf32de71f3a2d18674c1e7d3027f8d78c8c11244476a2dbd",
    "d617a58bc786df58b102479074eed60b9b074eb59fba8a33a8ff6b3d6e383129",
    "8d33cf4348b7c4fe539a4d7969ad421b6a40812616d2d196a141de02da28c04b"
  ]
}
